## Tests — LLM Categorization (Cache + MCC fallback)

Validates `data_processing/categorize_core.py` behavior without calling OpenAI.

Covers:
- deterministic fingerprinting
- cache hit avoids repeated LLM calls
- LLM low-confidence triggers MCC fallback
- LLM exception triggers MCC fallback


## Imports

In [1]:
from __future__ import annotations

import sys
import tempfile
from pathlib import Path

from diskcache import Cache


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data").is_dir() and (candidate / "artifacts").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate project root containing data/ and artifacts/")


ROOT = find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from data_processing.categorize_core import (
    categorize_with_cache_and_fallback,
    merchant_fingerprint,
    transaction_fingerprint,
)
from model.analytics_core import mcc_to_category


## Fixtures

In [2]:
tx1 = {
    "id": "1",
    "client_id": "10",
    "transaction_dt": "2010-01-01 00:00:00",
    "amount_usd": 12.34,
    "mcc_code": "5812",
    "mcc_description": "Eating Places and Restaurants",
    "merchant_city": "Dallas",
    "merchant_state": "TX",
}
tx2 = {
    "id": "2",
    "client_id": "10",
    "transaction_dt": "2010-01-02 00:00:00",
    "amount_usd": 86.45,
    "mcc_code": "5541",
    "mcc_description": "Service Stations",
    "merchant_city": "Merritt Island",
    "merchant_state": "FL",
}


## Fingerprint stability

In [3]:
fp_a = transaction_fingerprint(tx1)
fp_b = transaction_fingerprint(dict(tx1))
assert fp_a == fp_b
assert isinstance(fp_a, str) and len(fp_a) == 64

mf_a = merchant_fingerprint(tx1)
mf_b = merchant_fingerprint(dict(tx1))
assert mf_a == mf_b
assert isinstance(mf_a, str) and len(mf_a) == 64


## Cache + fallback behavior

In [4]:
calls = {"n": 0}

def llm_stub(batch):
    calls["n"] += 1
    # tx1: high confidence; tx2: low confidence to force MCC fallback
    return [
        {"id": "1", "category": "Dining", "confidence": 0.9},
        {"id": "2", "category": "Transportation", "confidence": 0.2},
    ]

with tempfile.TemporaryDirectory() as d:
    cache = Cache(d)
    out1 = categorize_with_cache_and_fallback(
        [tx1, tx2],
        llm_call=llm_stub,
        cache=cache,
        confidence_threshold=0.6,
    )
    assert calls["n"] == 1
    by_id = {r.transaction_id: r for r in out1}
    assert by_id["1"].category_final == "Dining"
    assert by_id["1"].source == "llm"

    # low-confidence forces deterministic MCC mapping
    assert by_id["2"].source == "low_confidence_fallback"
    assert by_id["2"].category_final == mcc_to_category(tx2["mcc_code"], tx2["mcc_description"])

    # second run should hit cache (no new LLM call)
    out2 = categorize_with_cache_and_fallback(
        [tx1, tx2],
        llm_call=llm_stub,
        cache=cache,
        confidence_threshold=0.6,
    )
    assert calls["n"] == 1
    assert {r.transaction_id for r in out2} == {"1", "2"}
    cache.close()

# Merchant-key caching: two different tx ids with same merchant should hit cache
calls2 = {"n": 0}

def llm_stub2(batch):
    calls2["n"] += 1
    return [{"id": str(t["id"]), "category": "Dining", "confidence": 0.9} for t in batch]

tx1b = dict(tx1)
tx1b["id"] = "1b"
tx1b["transaction_dt"] = "2010-01-05 00:00:00"

with tempfile.TemporaryDirectory() as d:
    cache = Cache(d)
    _ = categorize_with_cache_and_fallback(
        [tx1],
        llm_call=llm_stub2,
        cache=cache,
        cache_key_fn=merchant_fingerprint,
    )
    _ = categorize_with_cache_and_fallback(
        [tx1b],
        llm_call=llm_stub2,
        cache=cache,
        cache_key_fn=merchant_fingerprint,
    )
    assert calls2["n"] == 1
    cache.close()

# LLM exception triggers MCC fallback
def llm_fail(_batch):
    raise RuntimeError("boom")

with tempfile.TemporaryDirectory() as d:
    cache = Cache(d)
    out3 = categorize_with_cache_and_fallback([tx1], llm_call=llm_fail, cache=cache)
    assert out3[0].source == "mcc_fallback"
    assert out3[0].category_final == mcc_to_category(tx1["mcc_code"], tx1["mcc_description"])
    cache.close()

print("Categorize tests: PASS")


Categorize tests: PASS


In [5]:
import os

from dotenv import load_dotenv
from openai import OpenAI

# override=True: empty shell LLM_API_KEY="" must not block the real .env value
load_dotenv(ROOT / ".env", override=True)

API_KEY = (os.environ.get("LLM_API_KEY") or os.environ.get("OPENAI_API_KEY") or "").strip()
LLM_MODEL = os.environ.get("LLM_MODEL", "gpt-4o-mini")
BASE_URL = os.environ.get("OPENAI_BASE_URL") or os.environ.get("LLM_API_BASE")

assert API_KEY, (
    f"Missing LLM_API_KEY / OPENAI_API_KEY after loading {(ROOT / '.env')}. "
    "Re-run the Imports cell first, then confirm .env has a non-empty key."
)
print("Model:", LLM_MODEL)
print("Base URL:", BASE_URL or "(default OpenAI)")
print("API key loaded:", True, f"(len={len(API_KEY)})")

client_kwargs = {"api_key": API_KEY}
if BASE_URL:
    client_kwargs["base_url"] = BASE_URL
client = OpenAI(**client_kwargs)

# Tiny connection probe — not a full categorize request
try:
    resp = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": "hello"}],
        max_tokens=16,
        temperature=0,
    )
except TypeError:
    resp = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": "hello"}],
        max_completion_tokens=16,
        temperature=0,
    )

reply = (resp.choices[0].message.content or "").strip()
assert reply, repr(resp)
print("OpenAI connection: PASS")
print("Reply:", repr(reply))


Model: gpt-4o-mini
Base URL: https://aibe.mygreatlearning.com/openai/v1
API key loaded: True (len=67)


RateLimitError: Error code: 429 - {'reason': {'error': 'You exceeded your current quota!!'}}